In [0]:
from pyspark.sql.functions import current_timestamp, col

In [0]:
BASE = "/Volumes/databricksbootcamp/aerosurf/blueguard"

BRONZE_TABLE = "databricksbootcamp.aerosurf.raw_inspections"

SCHEMA_PATH = f"{BASE}/schema"
CHECKPOINT_PATH = f"{BASE}/checkpoint"
SOURCE_PATH = f"{BASE}/inspections"

In [0]:
bronze_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(SOURCE_PATH)
        .withColumn("bronze_ingested_at", current_timestamp())
)

In [0]:
(
    bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

In [0]:
display(spark.read.table("databricksbootcamp.aerosurf.raw_inspections"))

In [0]:
%sql
-- DROP TABLE databricksbootcamp.aerosurf.raw_inspections